# Experiment 55 - Exp22 Rank-Space Blending

**Building on:** exp53/54 plateau around OOF cmAP=0.9497.

**Hypothesis:** Padded cmAP is ranking-based, while the current blend is sensitive to component logit scale, especially the high-lambda prior. Test per-class percentile-rank blending of ProtoSSM, MLP, and prior components to reduce calibration artifacts while preserving ranking signal.


In [ ]:
import json
from pathlib import Path

nb_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
exp49_path = nb_dir / 'exp49_exp22_public_aware.ipynb'
exp49_nb = json.loads(exp49_path.read_text())

for cell_idx in range(1, 10):
    print(f'--- executing exp49 cell {cell_idx} ---')
    src = ''.join(exp49_nb['cells'][cell_idx]['source'])
    exec(compile(src, f'exp49_cell_{cell_idx}', 'exec'), globals())


In [ ]:
# Exp55: rank-space component blending

def percentile_rank_cols(x):
    order = np.argsort(x, axis=0, kind='mergesort')
    ranks = np.empty_like(order, dtype=np.float32)
    col = np.arange(x.shape[1])
    ranks[order, col] = np.arange(x.shape[0], dtype=np.float32)[:, None]
    return ranks / max(float(x.shape[0] - 1), 1.0)

LAMBDAS_EXT = [3.0, 4.0, 5.0]
TTA_W0S = [0.85, 0.90]
ENSEMBLE_GRIDS = [
    (0.02, 0.58, 0.40),
    (0.02, 0.63, 0.35),
    (0.05, 0.55, 0.40),
    (0.05, 0.60, 0.35),
    (0.00, 0.60, 0.40),
    (0.10, 0.55, 0.35),
]
POST_POWERS = [0.0, 0.5, 1.0]
POST_ALPHAS = [0.0, 0.05, 0.10]

best_cmap = 0.0
best_cfg = {}
results = []
rank_cache = {}

def get_rank_components(lam, tta_w0):
    key = (lam, tta_w0)
    if key not in rank_cache:
        prior_0 = apply_prior(oof_raw_0, oof_meta_sites, oof_meta_hours, prior_tables, lam,
                              prior_shrink=1.0, prior_logit_clip=None)
        prior_250 = apply_prior(oof_raw_250, oof_meta_sites, oof_meta_hours, prior_tables, lam,
                                prior_shrink=1.0, prior_logit_clip=None)
        proto = tta_w0 * oof_proto_0 + (1.0 - tta_w0) * oof_proto_250
        mlp = tta_w0 * oof_mlp_0 + (1.0 - tta_w0) * oof_mlp_250
        prior = tta_w0 * prior_0 + (1.0 - tta_w0) * prior_250
        rank_cache[key] = (
            percentile_rank_cols(proto),
            percentile_rank_cols(mlp),
            percentile_rank_cols(prior),
        )
    return rank_cache[key]

for lam in LAMBDAS_EXT:
    for tta_w0 in TTA_W0S:
        r_proto, r_mlp, r_prior = get_rank_components(lam, tta_w0)
        for wp, wm, wpr in ENSEMBLE_GRIDS:
            rank_score = wp * r_proto + wm * r_mlp + wpr * r_prior
            for power in POST_POWERS:
                for alpha in POST_ALPHAS:
                    probs = rank_score.copy()
                    if power > 0:
                        probs = file_confidence_scale(probs, power=power)
                    if alpha > 0:
                        probs = adaptive_delta_smooth(probs, base_alpha=alpha)
                    probs = np.clip(probs, 0.0, 1.0)
                    cmap = padded_cmap(Y_FULL_aligned, probs)
                    row = {'lam': lam, 'tta_w0': tta_w0, 'power': power, 'alpha': alpha,
                           'wp': wp, 'wm': wm, 'wpr': wpr, 'cmap': cmap}
                    results.append(row)
                    if cmap > best_cmap:
                        best_cmap = cmap
                        best_cfg = row.copy()

df = pd.DataFrame(results).sort_values('cmap', ascending=False).reset_index(drop=True)
print(f'Best OOF cmAP: {best_cmap:.5f}')
print(f'Best config:   {best_cfg}')
print('\nTop 25 configs:')
print(df.head(25).to_string(index=False))
print('\nBest cmAP by W_PRIOR:')
print(df.groupby('wpr')['cmap'].max().reset_index().to_string(index=False))

r_proto, r_mlp, r_prior = get_rank_components(best_cfg['lam'], best_cfg['tta_w0'])
p_best = best_cfg['wp'] * r_proto + best_cfg['wm'] * r_mlp + best_cfg['wpr'] * r_prior
if best_cfg['power'] > 0:
    p_best = file_confidence_scale(p_best, power=best_cfg['power'])
if best_cfg['alpha'] > 0:
    p_best = adaptive_delta_smooth(p_best, base_alpha=best_cfg['alpha'])
p_best = np.clip(p_best, 0.0, 1.0)
auc_b = macro_auc(Y_FULL_aligned, p_best)
cmap_b = padded_cmap(Y_FULL_aligned, p_best)

print('=' * 60)
print('Experiment 55 - Exp22 rank-space blending')
print(f"Best: lambda={best_cfg['lam']}, tta_w0={best_cfg['tta_w0']}, power={best_cfg['power']}, alpha={best_cfg['alpha']}")
print(f"Weights: wp={best_cfg['wp']}, wm={best_cfg['wm']}, wpr={best_cfg['wpr']}")
print(f'AUC={auc_b:.5f}  cmAP={cmap_b:.5f}')
print(f'vs exp53 OOF (0.94973): delta cmAP = {cmap_b - 0.94973:+.5f}')
print(f'vs exp22 OOF (0.93894): delta cmAP = {cmap_b - 0.93894:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')
